# Notebook 6 — Backtest Integrado: MPT + PMPT + Black-Litterman

**TCC — Pedro Augusto Pinheiro Reis · Ciências Contábeis · Universidade Federal de Goiás (UFG)**
*Moderna Teoria das Carteiras no Mercado de Ações Brasileiro: Comparação entre Otimizadores e Inputs*
Orientador: Prof. Dr. Moisés Ferreira da Cunha

| Metadado | Valor |
| :--- | :--- |
| Autor | Pedro Augusto Pinheiro Reis · *(e-mail a preencher)* |
| Instituição | FACE/UFG — Ciências Contábeis |
| Data da análise | *(preencher)* |
| Insumo | `data/Retornos/retornos_simples_saneado` (NB04), `rf_diario` (NB02), `ibov_retornos` (NB03) |
| Saídas | `strategy_returns.csv`, `desempenho_estrategias.csv`, `pesos_historico.csv` |
| Licença | MIT *(sugerida)* |

---

## Início — o backtest comparativo completo

Este é o notebook central de resultados: simula *out-of-sample*, na **mesma janela expansiva** e sob as
**mesmas fricções** (custo, deriva, vedação ao *look-ahead*), as **15 estratégias** que materializam a
comparação do título — três arcabouços lado a lado:

- **MPT (6):** EqualWeight, InvVol, MinVar, MinVar_c10, MaxSharpe, MaxSharpe_c10
- **PMPT (5):** MinCVaR, MaxSortino, MaxOmega, MaxKappa3, MinCDaR
- **Black-Litterman (4):** BL_classico, BL_classico_c10, BL_downside, BL_downside_c10

Ao rodar todas no mesmo arnês, qualquer diferença de desempenho reflete o **otimizador**, não diferenças
de implementação — condição para a comparação ser válida.

**Decisões fixadas (marcadas `DECISÃO`):**
- **Janela expansiva** com *warm-up* de 60 meses (1º rebalanceamento ≈ jan/2015), conforme Capítulo 3.
- **Black-Litterman com $w_{mkt}=1/N$** (prior neutro, sem *look-ahead*). A ponderação por capitalização
  própria (preço × nº de ações da CVM) fica registrada como **refinamento futuro**.
- **Visões do BL por momentum 12-1** (*placeholder*). O **CatBoost** entrará pela mesma interface
  `visoes_fn`, sem alterar o backtest.
- Dois priors do BL: **clássico** ($\Sigma$, CAPM) e **downside** ($\Sigma_D$ de Estrada, D-CAPM),
  para medir a inconsistência CAPM/PMPT também no desempenho.

### Sumário
0. Dependências · 1. Parâmetros · 2. Insumos · 3. Estimadores ($\Sigma$, $\Sigma_D$) ·
4. Otimizadores MPT · 5. Otimizadores PMPT · 6. Black-Litterman · 7. Validação ·
8. `construir_alvos` (15 estratégias) · 9. Backtest (janela expansiva) · 10. Métricas ·
11. Exportação · 12. Fechamento

## 0. Dependências e reprodutibilidade (Regra 5)

In [ ]:
import sys
import numpy as np, pandas as pd, scipy
from scipy.optimize import minimize
try:
    import cvxpy as cp
    CVXPY_OK = True; _SOLVERS=[s for s in ("CLARABEL","ECOS","SCS") if s in cp.installed_solvers()]
except Exception as e:
    CVXPY_OK=False; _SOLVERS=[]
    print(f"[AVISO] cvxpy ausente ({e}); MinCVaR/MinCDaR serão pulados. Instale: pip install cvxpy")
SEED=42; np.random.seed(SEED)
print("Python", sys.version.split()[0], "| numpy", np.__version__, "| pandas", pd.__version__,
      "| scipy", scipy.__version__, "| cvxpy", (cp.__version__ if CVXPY_OK else "AUSENTE"), _SOLVERS)

## 1. Parâmetros

In [ ]:
from pathlib import Path
parent_path  = Path.cwd().parent.parent
DIR_RETORNOS = parent_path / "data" / "Retornos"
OUTPUT_DIR   = parent_path / "data" / "Estrategias"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TRADING_DAYS = 252
WARMUP_MESES = 60          # treino inicial de 5 anos -> 1º rebal. ~jan/2015 (Cap. 3)
REBAL        = "M"
LONG_ONLY    = True
TETO_PESO    = 0.10        # CVM 175
CUSTO_BPS    = 50.0        # DECISÃO: Cap.3 = 0,5%; código original = 20bps. Reconciliar
ALPHA        = 0.95        # CVaR/CDaR
MAR_MODO     = "cdi"       # limiar do Kappa/Sortino/Omega
KAPPA_ORDENS = {"MaxOmega":1, "MaxSortino":2, "MaxKappa3":3}
# Black-Litterman
DELTA        = 2.5         # aversão ao risco (He-Litterman); δ implícito deu negativo (NB07)
TAU          = 0.05
MAR_ESTRADA  = "media"
VISAO_MOMENTUM_MESES = (12, 1)
# DECISÃO: w_mkt = 1/N (neutro, sem look-ahead). Capitalização própria = trabalho futuro.
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")
print(f"[OK] janela=expansiva | warmup={WARMUP_MESES}m | custo={CUSTO_BPS}bps | "
      f"teto={TETO_PESO:.0%} | alpha={ALPHA} | BL: δ={DELTA}, τ={TAU}, w_mkt=1/N")

## 2. Insumos

Retornos saneados (NB04), CDI (NB02), IBOVESPA (NB03). Base de cotações proprietária (Economática); ver `Datasets.md`.

In [ ]:
def _ler(nome, col=None):
    pq, csv = DIR_RETORNOS/f"{nome}.parquet", DIR_RETORNOS/f"{nome}.csv"
    df = (pd.read_parquet(pq) if pq.exists() else pd.read_csv(csv, index_col=0, parse_dates=True))
    df.index = pd.to_datetime(df.index); return df[col] if col else df

ret  = _ler("retornos_simples_saneado").sort_index()
ibov = _ler("ibov_retornos", "ibov_ret_simples").sort_index()
rf   = _ler("rf_diario", "cdi_diario").sort_index()
N = ret.shape[1]; tickers=[c.replace("ACAO_","") for c in ret.columns]
print(f"Retornos: {ret.shape[0]} pregões × {N} ativos "
      f"({ret.index.min().date()} → {ret.index.max().date()})")

## 3. Estimadores de risco: $\Sigma$ (Ledoit-Wolf) e $\Sigma_D$ (Estrada)

$\Sigma$ alimenta MPT e o BL clássico; $\Sigma_D=M^\top M/T$ (PSD por construção) alimenta o BL
*downside*. Sob janela expansiva, o peso de encolhimento de Ledoit-Wolf decresce com $T$ (documentar).

In [ ]:
def ledoit_wolf(X):
    X=np.asarray(X,float); T,n=X.shape; Xc=X-X.mean(0); S=(Xc.T@Xc)/T
    m=np.trace(S)/n; F=m*np.eye(n); d2=np.sum((S-F)**2)/n
    sq=(Xc**2).sum(1); quad=np.einsum('ti,ij,tj->t',Xc,S,Xc)
    b2=(np.sum(sq**2)-2*np.sum(quad)+T*np.sum(S**2))/(n*T**2)
    delta=min(b2,d2)/d2 if d2>0 else 0.0; return delta*F+(1-delta)*S

def estrada_semicov(X, mar="media"):
    X=np.asarray(X,float); T,n=X.shape
    tau = X.mean(0) if mar=="media" else (np.zeros(n) if mar=="zero" else np.full(n,float(mar)))
    M=np.minimum(X-tau,0.0); return (M.T@M)/T

## 4. Otimizadores MPT

In [ ]:
_sum=({"type":"eq","fun":lambda w: w.sum()-1.0},)
def _bounds(teto=None): hi=teto if teto is not None else 1.0; return [(0.0,hi)]*N if LONG_ONLY else [(-hi,hi)]*N
def w_equal(): return np.repeat(1.0/N,N)
def w_inv_vol(S): iv=1.0/np.sqrt(np.diag(S)); return iv/iv.sum()
def w_min_var(S, teto=None):
    r=minimize(lambda w:w@S@w, w_equal(), method="SLSQP", bounds=_bounds(teto),
               constraints=_sum, options={"maxiter":300,"ftol":1e-10}); return r.x/r.x.sum()
def w_max_sharpe(mu,S,rf_a,teto=None):
    def neg(w): v=np.sqrt(w@S@w); return -((w@mu-rf_a)/v) if v>0 else 0.0
    r=minimize(neg, w_equal(), method="SLSQP", bounds=_bounds(teto),
               constraints=_sum, options={"maxiter":500,"ftol":1e-10}); return r.x/r.x.sum()

## 5. Otimizadores PMPT (CVaR · Kappa/Sortino/Omega · CDaR)

In [ ]:
def _solve(prob):
    for s in _SOLVERS:
        try:
            prob.solve(solver=getattr(cp,s))
            if prob.status in ("optimal","optimal_inaccurate"): return True
        except Exception: continue
    return False

def w_max_kappa(janela, n=2, mar=0.0, teto=None):
    janela=np.asarray(janela,float)
    def neg(w):
        rp=janela@w; ex=rp-mar; lpm=np.mean(np.clip(mar-rp,0.0,None)**n)
        if lpm<=1e-18: return 0.0 if ex.mean()<=0 else -1e6
        return -ex.mean()/(lpm**(1.0/n))
    r=minimize(neg, w_equal(), method="SLSQP", bounds=_bounds(teto),
               constraints=_sum, options={"maxiter":500,"ftol":1e-12}); return r.x/r.x.sum()

def w_min_cvar(cen, alpha=0.95, teto=None):
    if not CVXPY_OK: raise RuntimeError("cvxpy")
    Rc=np.asarray(cen,float); T,k=Rc.shape
    w=cp.Variable(k); z=cp.Variable(); u=cp.Variable(T,nonneg=True); hi=teto if teto is not None else 1.0
    cons=[cp.sum(w)==1, u>=(-Rc@w)-z]+([w>=0,w<=hi] if LONG_ONLY else [])
    if not _solve(cp.Problem(cp.Minimize(z+(1/((1-alpha)*T))*cp.sum(u)),cons)): raise RuntimeError("CVaR")
    x=np.clip(np.asarray(w.value).flatten(),0,None); return x/x.sum()

def w_min_cdar(cen, alpha=0.95, teto=None):
    if not CVXPY_OK: raise RuntimeError("cvxpy")
    Rc=np.asarray(cen,float); T,k=Rc.shape; Rcum=np.cumsum(Rc,axis=0)
    w=cp.Variable(k); u=cp.Variable(T); zv=cp.Variable(T,nonneg=True); z=cp.Variable()
    C=Rcum@w; hi=teto if teto is not None else 1.0
    cons=[cp.sum(w)==1, u>=C, u[1:]>=u[:-1], u[0]>=0, zv>=(u-C)-z]+([w>=0,w<=hi] if LONG_ONLY else [])
    if not _solve(cp.Problem(cp.Minimize(z+(1/((1-alpha)*T))*cp.sum(zv)),cons)): raise RuntimeError("CDaR")
    x=np.clip(np.asarray(w.value).flatten(),0,None); return x/x.sum()

## 6. Black-Litterman (dois priors, visões plugáveis)

$w_{mkt}=1/N$. Visões absolutas por momentum 12-1 (P=I), com $\Omega$ no padrão He-Litterman. A fórmula
mestra é aplicada a cada prior. **Slot do CatBoost:** trocar `visoes_momentum` por `visoes_boosting`.

In [ ]:
def visoes_momentum(janela, tau=TAU, Sigma=None):
    R=np.asarray(janela,float); T,n=R.shape; L,S_=VISAO_MOMENTUM_MESES
    jl,js=int(L*21),int(S_*21)
    if T<jl+5: Q=R.mean(0)*TRADING_DAYS
    else:
        bloco=R[-jl:-js] if js>0 else R[-jl:]
        Q=np.prod(1+bloco,axis=0)**(TRADING_DAYS/len(bloco))-1
    P=np.eye(n)
    Sg=Sigma if Sigma is not None else ledoit_wolf(R)*TRADING_DAYS
    Om=np.diag(np.maximum(np.diag(P@(tau*Sg)@P.T),1e-8))
    return P,Q,Om

def bl_posterior(Sigma, Pi, P, Q, Omega, tau=TAU):
    tauS_inv=np.linalg.inv(tau*Sigma); Om_inv=np.linalg.inv(Omega)
    A=tauS_inv+P.T@Om_inv@P; b=tauS_inv@Pi+P.T@Om_inv@Q
    return np.linalg.solve(A,b)

## 7. Validação dos otimizadores (caso controlado, antes do backtest)

In [ ]:
def _cvar_emp(rp,a=0.95): p=-np.asarray(rp); v=np.quantile(p,a); c=p[p>=v]; return c.mean() if len(c) else v
_rng=np.random.default_rng(SEED); T_=1500
_R=np.column_stack([_rng.normal(0.0008,0.010,T_), _rng.normal(0.0004,0.014,T_), _rng.normal(0.0005,0.012,T_)])
_R[_rng.choice(T_,30,replace=False),2]-=0.08
_S=ledoit_wolf(_R); _SD=estrada_semicov(_R)
_N_bak=N; globals()['N']=3
def _inv(nm,w): assert abs(w.sum()-1)<1e-6 and (w>=-1e-9).all(), nm; return w
_inv("MinVar",w_min_var(_S)); _inv("MaxSharpe",w_max_sharpe(_R.mean(0)*252,_S,0.0))
for _nm,_n in KAPPA_ORDENS.items(): _inv(_nm,w_max_kappa(_R,n=_n))
# BL sanity: Ω→∞ ⇒ Π ; Ω→0 ⇒ Q ; reverse-opt ⇒ w_mkt
_wm=np.repeat(1/3,3); _Pi=DELTA*_S@_wm; _P,_Q,_Om=visoes_momentum(_R,TAU,_S)
assert np.allclose(bl_posterior(_S,_Pi,_P,_Q,np.diag([1e6]*3)),_Pi,atol=1e-4)
assert np.allclose(bl_posterior(_S,_Pi,_P,_Q,np.diag([1e-10]*3)),_Q,atol=1e-3)
assert np.allclose((1/DELTA)*np.linalg.solve(_S,_Pi),_wm,atol=1e-8)
if CVXPY_OK:
    assert _cvar_emp(_R@w_min_cvar(_R,ALPHA))<=_cvar_emp(_R@w_equal())+1e-9
globals()['N']=_N_bak
print("[OK] MPT, PMPT e BL validados (invariantes + sanity checks analíticos).")

## 8. `construir_alvos` — as 15 estratégias

Contrato único: recebe a janela e os estimadores, devolve o vetor de pesos de cada estratégia
(soma 1, *long-only*). As convexas (CVaR/CDaR) só entram se `cvxpy` existir.

In [ ]:
def _mar_d(rf_j): return float(rf_j.mean()) if MAR_MODO=="cdi" else 0.0

def construir_alvos(janela, S, SigD, mu, rf_a, mar_d):
    a = {
        "EqualWeight":   w_equal(),
        "InvVol":        w_inv_vol(S),
        "MinVar":        w_min_var(S, None),
        "MinVar_c10":    w_min_var(S, TETO_PESO),
        "MaxSharpe":     w_max_sharpe(mu, S, rf_a, None),
        "MaxSharpe_c10": w_max_sharpe(mu, S, rf_a, TETO_PESO),
    }
    for nm,n in KAPPA_ORDENS.items():
        a[nm] = w_max_kappa(janela, n=n, mar=mar_d, teto=None)
    if CVXPY_OK:
        a["MinCVaR"] = w_min_cvar(janela, ALPHA, None)
        a["MinCDaR"] = w_min_cdar(janela, ALPHA, None)
    # --- Black-Litterman: 2 priors × {sem teto, c10} ---
    wm = np.repeat(1.0/N, N)
    P, Q, Om = visoes_momentum(janela, TAU, S)
    for nome, Sg, Pi in [("classico", S, DELTA*S@wm), ("downside", SigD, DELTA*SigD@wm)]:
        mu_bl = bl_posterior(Sg, Pi, P, Q, Om, TAU)
        a[f"BL_{nome}"]     = w_max_sharpe(mu_bl, Sg, rf_a, None)
        a[f"BL_{nome}_c10"] = w_max_sharpe(mu_bl, Sg, rf_a, TETO_PESO)
    return a

ESTRATEGIAS = (["EqualWeight","InvVol","MinVar","MinVar_c10","MaxSharpe","MaxSharpe_c10"]
               + list(KAPPA_ORDENS.keys())
               + (["MinCVaR","MinCDaR"] if CVXPY_OK else [])
               + ["BL_classico","BL_classico_c10","BL_downside","BL_downside_c10"])
print(f"{len(ESTRATEGIAS)} estratégias:\n  ", ESTRATEGIAS)

## 9. Backtest com janela expansiva, deriva e custos

A cada mês (após 60 de *warm-up*): janela = toda a história até o fim do mês anterior; estima $\Sigma$ e
$\Sigma_D$; constrói as 15 carteiras; aplica os retornos do mês com custo de `CUSTO_BPS` sobre o
*turnover* (no dia do rebalanceamento) e deriva de pesos intra-mês.

> Runtime: 15 estratégias × ~132 rebalanceamentos, com CVaR/CDaR convexos crescendo com $T$. Pode levar
> **vários minutos**.

In [ ]:
periodos  = ret.index.to_period(REBAL); uper = pd.Index(periodos.unique())
ret_estr={k:[] for k in ESTRATEGIAS}; pesos_hist={k:{} for k in ESTRATEGIAS}
turn_hist={k:[] for k in ESTRATEGIAS}; w_ant={k:None for k in ESTRATEGIAS}
custo=CUSTO_BPS/1e4

for i in range(WARMUP_MESES, len(uper)):
    dias=ret.index[periodos==uper[i]]
    fim_prev=ret.index[periodos==uper[i-1]][-1]
    janela=ret.loc[:fim_prev]
    if janela.shape[0] < WARMUP_MESES*21: continue
    Jv=janela.values
    S=ledoit_wolf(Jv)*TRADING_DAYS
    SigD=estrada_semicov(Jv, MAR_ESTRADA)*TRADING_DAYS
    mu=janela.mean().values*TRADING_DAYS
    rf_j=rf.reindex(janela.index).dropna(); rf_a=rf_j.mean()*TRADING_DAYS; mar_d=_mar_d(rf_j)
    alvos=construir_alvos(Jv, S, SigD, mu, rf_a, mar_d)
    Rm=ret.loc[dias].values
    for k,w in alvos.items():
        turn=1.0 if w_ant[k] is None else np.abs(w-w_ant[k]).sum()
        turn_hist[k].append((dias[0],turn))
        rp=(Rm@w).astype(float).copy(); rp[0]-=custo*turn
        ret_estr[k].append(pd.Series(rp,index=dias)); pesos_hist[k][dias[0]]=w
        cres=np.prod(1+Rm,axis=0); wd=w*cres; w_ant[k]=wd/wd.sum()

strategy_returns=pd.DataFrame({k:pd.concat(v) for k,v in ret_estr.items()})
strategy_returns["IBOV"]=ibov.reindex(strategy_returns.index); strategy_returns.index.name="data"
print(f"OOS: {strategy_returns.shape[0]} pregões | {strategy_returns.index.min().date()} → "
      f"{strategy_returns.index.max().date()} | {len(turn_hist[ESTRATEGIAS[0]])} rebalanceamentos")

## 10. Métricas de desempenho

In [ ]:
def metricas(r, rf_s):
    r=r.dropna(); rf_m=rf_s.reindex(r.index).mean(); cum=(1+r).cumprod()
    cagr=cum.iloc[-1]**(TRADING_DAYS/len(r))-1; vol=r.std()*np.sqrt(TRADING_DAYS)
    sharpe=(r.mean()-rf_m)/r.std()*np.sqrt(TRADING_DAYS)
    exc=r-rf_m; dn=np.sqrt((exc.clip(upper=0)**2).mean())
    sortino=exc.mean()/dn*np.sqrt(TRADING_DAYS) if dn>0 else np.nan
    return {"ret_anual":cagr,"vol_anual":vol,"sharpe":sharpe,"sortino":sortino,
            "max_drawdown":(cum/cum.cummax()-1).min()}
desempenho=pd.DataFrame({c:metricas(strategy_returns[c],rf) for c in strategy_returns.columns}).T
for k in ESTRATEGIAS: desempenho.loc[k,"turnover_aa"]=np.mean([v for _,v in turn_hist[k][1:]])*12
desempenho.loc["IBOV","turnover_aa"]=np.nan
print("Desempenho out-of-sample (líquido de custos):\n"); print(desempenho.round(4).to_string())

## 11. Exportação

In [ ]:
def _salvar(df,nome):
    df.to_csv(OUTPUT_DIR/f"{nome}.csv", date_format="%Y-%m-%d", float_format="%.8f")
    try: df.to_parquet(OUTPUT_DIR/f"{nome}.parquet"); print(f"  {nome}.csv + .parquet")
    except Exception as e: print(f"  {nome}.csv [{e}]")
print("[+] Exportando em", OUTPUT_DIR, "\n")
_salvar(strategy_returns,"strategy_returns"); _salvar(desempenho,"desempenho_estrategias")
linhas=[]
for k in ESTRATEGIAS:
    for data,w in pesos_hist[k].items():
        for tk,p in zip(tickers,w):
            if p>1e-6: linhas.append({"estrategia":k,"data":data,"ticker":tk,"peso":p})
pd.DataFrame(linhas).to_csv(OUTPUT_DIR/"pesos_historico.csv", index=False, float_format="%.6f")
print(f"  pesos_historico.csv ({len(linhas)} linhas)\n[OK] {len(ESTRATEGIAS)} estratégias → insumo do NB07")

## 12. Fechamento — interpretação

*(Preencher após execução — Regra 1.)*

- **MPT × PMPT × BL:** qual arcabouço entrega melhor relação risco-retorno ajustado (Sharpe/Sortino)
  líquido de custos? A vantagem teórica da PMPT/BL se confirma no mercado brasileiro?
- **Efeito do teto (CVM 175):** comparar cada estratégia com sua variante `_c10`.
- **BL clássico × downside:** a versão *downside* (Σ_D) supera a clássica? A diferença mede, no
  desempenho, a inconsistência CAPM/PMPT discutida no NB6c.
- **Comparabilidade:** CVaR/CDaR não usam Σ; BL usa Σ/Σ_D; documentar que parte da diferença vem do
  insumo, não só do objetivo.
- **Custo/turnover:** confrontar `turnover_aa` com o desempenho líquido (viabilidade operacional).

**Limitações:** Kappa/Sortino/Omega e MaxSharpe são não-convexos (SLSQP, ótimo local); BL com $w_{mkt}=1/N$
e visões por momentum (*placeholders* até capitalização própria e CatBoost); shrinkage de Ledoit-Wolf
enfraquece com $T$ na janela expansiva; teste formal de significância das diferenças → NB07.

---
# Autoavaliação — *Ten Simple Rules* (Rule et al., 2019)

| Regra | Tema | Status | Evidência / Ação |
| :---: | :--- | :---: | :--- |
| 1 | Contar uma história | 🟡 | Início/meio prontos; fim = seção 12, após execução |
| 2 | Documentar o processo | ✅ | Cabeçalho; decisões marcadas DECISÃO |
| 3 | Divisão clara de células | ✅ | Uma tarefa por célula; sumário |
| 4 | Modularizar código | 🟡 | Funções nomeadas; **ação:** `utils_pipeline.py` (LW/Estrada duplicados nos NBs) |
| 5 | Registrar dependências | 🟡 | Versões impressas; **ação:** `requirements.txt` |
| 6 | Controle de versão | ⚪ | Git + `nbdime` |
| 7 | Construir um pipeline | ✅ | Parâmetros no topo; consome NB02/03/04; alimenta NB07 |
| 8 | Compartilhar/explicar dados | 🟡 | Base proprietária; `Datasets.md` |
| 9 | Ler, rodar e explorar | ✅ | Linear; validação falha cedo; degrada sem cvxpy |
| 10 | Pesquisa aberta | 🟡 | Licença MIT + repositório sem dados |